In [6]:
#-------------------------------ArcGIS file {convert the X_Center, Y_Center values from UTM (meters) to Latitude/Longitude (degrees)} ----------------
import pandas as pd
import numpy as np
import pyproj

# Load the dataset
file_path = r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\I.Arcgis\x_y_z1.txt"
df_grid = pd.read_csv(file_path, delimiter="\t")

# Define UTM projection (Zone 15N) and WGS84 (lat/lon) projection
proj_utm = pyproj.Proj(init="epsg:32615")  # UTM Zone 15N (meters)
proj_wgs84 = pyproj.Proj(init="epsg:4326")  # WGS84 (Latitude, Longitude)

# Convert UTM (X_Center, Y_Center) to Latitude/Longitude (WGS84)
df_grid["Longitude"], df_grid["Latitude"] = pyproj.transform(proj_utm, proj_wgs84, 
                                                             df_grid["x"].values, 
                                                             df_grid["y"].values)

# ✅ Replace "<Null>" with actual NaN
df_grid.replace("<Null>", np.nan, inplace=True)

# Replace Null (NaN) values with 0
df_grid.fillna(0, inplace=True)

# Select necessary columns and save the updated dataset
output_file_path = r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\I.Arcgis\x_y_z.csv"
df_grid.to_csv(output_file_path, index=False)

# Display the updated dataset with Latitude and Longitude
print(f"File saved successfully at: {output_file_path}")
print(df_grid.head())  # Show first few rows


C:\Users\abuoliem\AppData\Local\anaconda3\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
C:\Users\abuoliem\AppData\Local\anaconda3\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
C:\Users\abuoliem\AppData\Local\Temp\ipykernel_4184\3483504995.py:15: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-p

File saved successfully at: C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\I.Arcgis\x_y_z.csv
   pointid  grid_code         x          y  Longitude   Latitude
0        1  -0.232072  482502.5  3348947.5 -93.181918  30.272070
1        2  -0.235828  482402.5  3348897.5 -93.182957  30.271617
2        3  -0.235828  482452.5  3348897.5 -93.182437  30.271618
3        4   0.706978  482502.5  3348897.5 -93.181917  30.271618
4        5  -0.228316  482552.5  3348897.5 -93.181397  30.271619


In [4]:
#-------------------------------               Assign feature values from the nearest (x, y) to (x', y') ----------------
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ------------------------------- Step 1: Load Data ------------------------------- #
# File paths (change as needed)
file1_path = r"C:\Users\abuoliem\Box\3.Third year2025\1.Fall25\3.Research\5.Phase1_Damage paper\K.final files\x'_and_y'.csv"  # File with x', y', y1
file2_path = r"C:\Users\abuoliem\Box\3.Third year2025\1.Fall25\3.Research\5.Phase1_Damage paper\K.final files\features2.csv"  # File with x, y, features

# Load datasets
df_new = pd.read_csv(file1_path)  # Load x', y', y1
df_ref = pd.read_csv(file2_path)  # Load x, y, and feature values

# Verify column names (Update based on your actual column names if different)
print("Columns in x',y':", df_new.columns)
print("Columns in final_grid_with_latlon:", df_ref.columns)

# ------------------------------- Step 2: Apply Longitude Shift ------------------------------- #
SHIFT_VALUE = -0.4  # Adjust this based on the actual difference
df_new["x'_corrected"] = df_new["x'"] + SHIFT_VALUE  # Apply correction

# ------------------------------- Step 3: Assign Feature Values ------------------------------- #
tree = cKDTree(df_ref[["x", "y"]])

# Find nearest neighbor
distances, indices = tree.query(df_new[["x'_corrected", "y'"]])

# Assign feature values from the matched data
for feature in ["BAR", "z", "VV_Pre", "VH_Pre", "VV_Post", "VH_Post", "POP_PRE", "POP_Pos", "NLCD_2019", "NLCD_2021"]:
    df_new[feature] = df_ref.iloc[indices][feature].values

# ------------------------------- Step 4: Save and Display Result ------------------------------- #
# Save merged dataset
output_file_path = r"C:\Users\abuoliem\Box\3.Third year2025\1.Fall25\3.Research\5.Phase1_Damage paper\K.final files\merged_x_prime_y_prime_with_features2.csv"
df_new.to_csv(output_file_path, index=False)

# Print confirmation and preview the first few rows
print(f"✅ Matching completed with longitude shift correction! File saved at: {output_file_path}")
print(df_new.head())  # Show first few rows

Columns in x',y': Index(['cell number', 'x'', 'y'', 'y1'], dtype='object')
Columns in final_grid_with_latlon: Index(['x', 'y', 'BAR', 'z', 'VV_Pre', 'VH_Pre', 'VV_Post', 'VH_Post',
       'POP_PRE', 'POP_Pos', 'NLCD_2019', 'NLCD_2021'],
      dtype='object')
✅ Matching completed with longitude shift correction! File saved at: C:\Users\abuoliem\Box\3.Third year2025\1.Fall25\3.Research\5.Phase1_Damage paper\K.final files\merged_x_prime_y_prime_with_features2.csv
   cell number       x'       y'    y1  x'_corrected        BAR          z  \
0            1 -92.7803  30.2287  0.15      -93.1803  27.086031  -0.296330   
1            2 -92.7803  30.2292  0.14      -93.1803  22.273393  -0.348429   
2            3 -92.7803  30.2296  0.13      -93.1803   7.704103  14.729088   
3            4 -92.7803  30.2301  0.13      -93.1803   0.003566  19.928968   
4            5 -92.7803  30.2305  0.15      -93.1803  21.922497  14.605096   

      VV_Pre     VH_Pre   VV_Post    VH_Post  POP_PRE  POP_Pos  NL

In [11]:
####Print Min/Max Ranges for Both Datasets###
print("Coordinate Ranges:")
print("x', y' File (New Data):")
print(f"  X' min: {df_new['x\''].min()}, X' max: {df_new['x\''].max()}")
print(f"  Y' min: {df_new['y\''].min()}, Y' max: {df_new['y\''].max()}")

print("\nFinal Grid with Lat/Lon File (Reference Data):")
print(f"  X min: {df_ref['x'].min()}, X max: {df_ref['x'].max()}")
print(f"  Y min: {df_ref['y'].min()}, Y max: {df_ref['y'].max()}")


Coordinate Ranges:
x', y' File (New Data):
  X' min: -92.82995414, X' max: -92.73581166
  Y' min: 30.20866273, Y' max: 30.24469487

Final Grid with Lat/Lon File (Reference Data):
  X min: -93.31669198, X max: -93.12903757
  Y min: 30.12452225, Y max: 30.27201336


In [10]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# Load the two Excel files
file1_path = r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\K.final files\x_y_z.csv"  # Update with the actual path
file2_path = r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\K.final files\features.csv"  # Update with the actual path

df1 = pd.read_csv(file1_path)  # First file (x_y_z)
df2 = pd.read_csv(file2_path)  # Second file (features)

# Rename columns for consistency
df1.rename(columns={"x": "X1", "y": "Y1", "z": "Z_1"}, inplace=True)
df2.rename(columns={"x": "X2", "y": "Y2", "z": "Z_2"}, inplace=True)

# Build a KDTree for fast nearest neighbor search
tree = cKDTree(df2[["X2", "Y2"]].values)

# Find the nearest neighbor for each (X1, Y1) in df2
distances, indices = tree.query(df1[["X1", "Y1"]].values)

# Add the nearest neighbor's values to df1
df1["X_Nearest"] = df2.iloc[indices]["X2"].values
df1["Y_Nearest"] = df2.iloc[indices]["Y2"].values
df1["Distance_to_Nearest"] = distances  # Euclidean distance between matched points

# Calculate coordinate differences
df1["X_Diff"] = abs(df1["X1"] - df1["X_Nearest"])
df1["Y_Diff"] = abs(df1["Y1"] - df1["Y_Nearest"])

# Print first few rows with coordinate differences
print("\nNearest Coordinate Differences:")
print(df1[["X1", "Y1", "X_Nearest", "Y_Nearest", "X_Diff", "Y_Diff", "Distance_to_Nearest"]].head())




Nearest Coordinate Differences:
          X1         Y1  X_Nearest  Y_Nearest    X_Diff    Y_Diff  \
0 -93.181918  30.272070 -93.182015  30.272013  0.000097  0.000056   
1 -93.182957  30.271617 -93.182838  30.271597  0.000119  0.000020   
2 -93.182437  30.271618 -93.182463  30.271648  0.000026  0.000030   
3 -93.181917  30.271618 -93.182013  30.271648  0.000096  0.000029   
4 -93.181397  30.271619 -93.181564  30.271648  0.000167  0.000028   

   Distance_to_Nearest  
0             0.000112  
1             0.000121  
2             0.000040  
3             0.000101  
4             0.000170  
